# MyOfis: uji AI open-source atas GPU Colab

Notebook ni uji sama ada model open-source (jalan dengan **Ollama**) cukup bagus untuk agent MyOfis:

1. **Chat**: jawab dalam BM / Manglish
2. **Tool calling**: agent MyOfis bergantung pada ni
3. **Vision**: baca gambar (untuk Publisher / Editor)
4. **Kelajuan**: token sesaat
5. *(Pilihan)* **Sambung ke MyOfis** sebagai provider "Custom (OpenAI-compatible)" untuk cuba sekejap

**Mula dulu:** Runtime → Change runtime type → **T4 GPU** → Save. Lepas tu Runtime → **Run all**.

> Colab percuma: sesi tamat bila idle atau lepas beberapa jam, dan URL tunnel bertukar setiap kali. Guna untuk **test sahaja**. Kalau model ni bagus, kita pasang yang sama di PC awak (RTX 3060 12 GB) supaya kekal.

In [ ]:
# Model to test. Both fit a free T4 (16 GB). Change them and Run all again to compare.
CHAT_MODEL = "qwen2.5:7b"     # good tool calling; try also "qwen3:8b" or "llama3.1:8b"
VISION_MODEL = "gemma3:4b"    # reads images; try also "qwen2.5vl:7b"
CONTEXT = 16384               # MyOfis prompts (tools + memory) are long: keep this >= 16k

!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Pasang dan hidupkan Ollama

In [ ]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import os, subprocess, time, requests
env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_CONTEXT_LENGTH=str(CONTEXT), OLLAMA_KEEP_ALIVE="30m")
ollama = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get("http://127.0.0.1:11434/api/version", timeout=2)
        break
    except Exception:
        time.sleep(1)
print("Ollama", requests.get("http://127.0.0.1:11434/api/version").json()["version"], "is running")

In [ ]:
# Download the models (a few GB, 1–3 minutes each)
!ollama pull {CHAT_MODEL}
!ollama pull {VISION_MODEL}

## 2. Uji chat (BM / Manglish)

In [ ]:
API = "http://127.0.0.1:11434/v1/chat/completions"

def chat(model, messages, **extra):
    t = time.time()
    r = requests.post(API, json={"model": model, "messages": messages, **extra}, timeout=600).json()
    r["_seconds"] = round(time.time() - t, 1)
    return r

r = chat(CHAT_MODEL, [
    {"role": "system", "content": "You are Marketing for SmashPoint, a badminton court in Malaysia. Reply in casual Malay (Manglish ok)."},
    {"role": "user", "content": "Tulis caption Facebook pendek untuk promo court RM15/jam hari Isnin-Rabu sebelum 5 petang. Ada call to action."},
])
print(r["choices"][0]["message"]["content"])
print(f"\n⏱ {r['_seconds']} s · {r['usage']['completion_tokens']} tokens")

## 3. Uji tool calling

Agent MyOfis tak boleh kerja tanpa ni. Model kena **panggil tool** dengan argumen yang betul, bukan berbual sahaja.

In [ ]:
import json

tools = [{
    "type": "function",
    "function": {
        "name": "read_bookings",
        "description": "Read SmashPoint court bookings for a date range (YYYY-MM-DD).",
        "parameters": {
            "type": "object",
            "properties": {"from": {"type": "string"}, "to": {"type": "string"}},
            "required": ["from"],
        },
    },
}]
r = chat(CHAT_MODEL, [
    {"role": "system", "content": "You are Kerani. Today is 2026-10-19. Use tools to get facts; never guess."},
    {"role": "user", "content": "Berapa slot kosong esok?"},
], tools=tools)
msg = r["choices"][0]["message"]
calls = msg.get("tool_calls") or []
if calls:
    for c in calls:
        print("✅ tool call:", c["function"]["name"], c["function"]["arguments"])
    args = json.loads(calls[0]["function"]["arguments"])
    print("   date right?", "✅" if args.get("from") == "2026-10-20" else f"❌ expected 2026-10-20, got {args.get('from')}")
else:
    print("❌ no tool call. The model answered with text instead:\n", msg.get("content"))

## 4. Uji vision (baca gambar poster)

In [ ]:
import base64, io
from PIL import Image, ImageDraw, ImageFont

# A fake event poster, drawn here so the test needs no upload
img = Image.new("RGB", (800, 500), (16, 120, 72))
d = ImageDraw.Draw(img)
try:
    big = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", 54)
    small = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", 34)
except OSError:
    big = small = ImageFont.load_default()
d.text((40, 60), "SMASHPOINT OPEN 2026", font=big, fill="white")
d.text((40, 180), "Sabtu 25 Oktober, 8:00 pagi", font=small, fill="white")
d.text((40, 240), "Yuran RM40 / pasangan", font=small, fill="white")
d.text((40, 300), "Hadiah utama RM500", font=small, fill=(255, 220, 0))
buf = io.BytesIO(); img.save(buf, format="PNG")
display(img.resize((400, 250)))

data_url = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()
r = chat(VISION_MODEL, [{"role": "user", "content": [
    {"type": "text", "text": "Read this poster. List: event name, date, time, fee, prize. Reply as short bullet points."},
    {"type": "image_url", "image_url": {"url": data_url}},
]}])
answer = r["choices"][0]["message"]["content"]
print(answer)
found = [w for w in ["SMASHPOINT", "25", "8", "40", "500"] if w.lower() in answer.lower()]
print(f"\n{'✅' if len(found) == 5 else '⚠️'} found {len(found)}/5 facts · ⏱ {r['_seconds']} s")

## 5. Kelajuan

In [ ]:
for model in [CHAT_MODEL, VISION_MODEL]:
    r = chat(model, [{"role": "user", "content": "Write about 200 words on why playing badminton is good for health."}], max_tokens=300)
    n = r["usage"]["completion_tokens"]
    print(f"{model:>16}: {n} tokens in {r['_seconds']} s ≈ {n / r['_seconds']:.1f} tokens/s")
print("\nAs a guide: above ~15 tokens/s feels fine in chat. An RTX 3060 is usually a bit faster than a Colab T4.")

## 6. (Pilihan) Sambung ke MyOfis untuk cuba sekejap

Cell ni buka URL awam sementara (Cloudflare quick tunnel), **dikunci dengan API key rawak**. Lepas tu dalam MyOfis:

1. **Settings → Providers → Add** → pilih **Custom (OpenAI-compatible)**
2. **Base URL**: yang dicetak di bawah (berakhir `/v1`)
3. **API key**: yang dicetak di bawah
4. Buka **Agents** → pilih agent → tukar provider + model (contoh `qwen2.5:7b`) → cuba chat

Bila habis test: padam provider tu, atau tukar agent balik ke canopy-ai. Lepas Colab ditutup, URL ni mati.

In [ ]:
import secrets, threading, re, http.server, urllib.request

API_KEY = "myofis-" + secrets.token_urlsafe(24)
UPSTREAM = "http://127.0.0.1:11434"

class AuthProxy(http.server.BaseHTTPRequestHandler):
    # Only requests with our key reach Ollama; the answer is streamed back as it comes (MyOfis streams).
    def _forward(self):
        if self.headers.get("Authorization") != f"Bearer {API_KEY}":
            self.send_response(401); self.send_header("Content-Type", "application/json"); self.end_headers()
            self.wfile.write(b'{"error":{"message":"unauthorized"}}'); return
        n = int(self.headers.get("Content-Length") or 0)
        body = self.rfile.read(n) if n else None
        req = urllib.request.Request(UPSTREAM + self.path, data=body, method=self.command,
                                     headers={"Content-Type": self.headers.get("Content-Type", "application/json")})
        try:
            r = urllib.request.urlopen(req, timeout=900)
        except urllib.error.HTTPError as e:
            r = e
        self.send_response(r.status if hasattr(r, "status") else r.code)
        self.send_header("Content-Type", r.headers.get("Content-Type", "application/json"))
        self.end_headers()
        while chunk := r.read1(8192) if hasattr(r, "read1") else r.read(8192):
            self.wfile.write(chunk); self.wfile.flush()
    do_GET = do_POST = _forward
    def log_message(self, *a): pass

server = http.server.ThreadingHTTPServer(("127.0.0.1", 8000), AuthProxy)
threading.Thread(target=server.serve_forever, daemon=True).start()

!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0); break

time.sleep(5)  # the new hostname takes a moment to work
ok = requests.get(url + "/v1/models", headers={"Authorization": f"Bearer {API_KEY}"}, timeout=30).status_code
print("Base URL :", url + "/v1")
print("API key  :", API_KEY)
print("Models   :", CHAT_MODEL, "/", VISION_MODEL)
print("Check    :", "✅ reachable" if ok == 200 else f"⚠️ HTTP {ok}, wait 10 s and run this cell's last lines again")
print("\nKeep this tab open while you test. Do not share the key.")

## Keputusan

| Ujian | Lulus kalau |
|---|---|
| Chat | BM / Manglish natural, ikut arahan |
| Tool calling | ✅ tool call dengan tarikh **2026-10-20** |
| Vision | jumpa 5/5 fakta poster |
| Kelajuan | > ~15 tokens/s |

Bagitahu Claude keputusan setiap ujian (screenshot pun boleh). Kalau lulus, kita tambah **Ollama di PC awak** sebagai provider tetap dan guna untuk kerja ringan (caption, taklimat, ringkasan, review), supaya jimat kos canopy-ai.